In [2]:
import pandas as pd

In [25]:
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "hiiragi12138/"
    "datascience-toolbox_group-project/"
    "main/"
    "instagram_analytics_cleaned.csv"
)

df = pd.read_csv(
    DATA_URL,
    parse_dates=[
        "post_datetime",
        "post_date"
    ]
)

df["account_id"] = df["account_id"].astype("category")

print(df.shape)
df.head()

(29999, 23)


,post_id,account_id,account_type,follower_count,media_type,content_category,traffic_source,has_call_to_action,post_datetime,post_date,...,comments,shares,saves,reach,impressions,engagement_rate,followers_gained,caption_length,hashtags_count,performance_bucket_label
0,IG0000001,7,brand,3551,reel,Technology,Home Feed,True,2024-11-30 06:00:00,2024-11-30,...,5,7,34,4327,6230,0.0385,899,100,7,medium
1,IG0000002,20,creator,31095,image,Fitness,Hashtags,True,2025-08-15 15:00:00,2025-08-15,...,10,21,68,7451,8268,0.0663,805,122,5,viral
2,IG0000003,15,brand,8167,reel,Beauty,Reels Feed,False,2025-09-11 16:00:00,2025-09-11,...,2,1,22,1639,2616,0.0531,758,115,8,high
3,IG0000004,11,creator,9044,carousel,Music,External,False,2025-09-18 03:00:00,2025-09-18,...,0,7,0,2877,3171,0.0309,402,115,7,medium
4,IG0000005,8,creator,15986,reel,Technology,Profile,False,2025-03-21 09:00:00,2025-03-21,...,8,5,21,5350,8503,0.0221,155,112,9,low


In [26]:
print(df["has_call_to_action"].value_counts())

has_call_to_action
False    19536
True     10463
Name: count, dtype: int64


## 1. 有無 CTA 的整體比較


In [27]:
overall = (
    df.groupby("has_call_to_action")
      .agg(
          posts=("post_id", "count"),
          avg_engagement_rate=("engagement_rate", "mean"),
          median_engagement_rate=("engagement_rate", "median"),
          avg_followers_gained=("followers_gained", "mean"),
          median_followers_gained=("followers_gained", "median"),
          avg_reach=("reach", "mean")
      )
      .reset_index()
)

display(overall)

,has_call_to_action,posts,avg_engagement_rate,median_engagement_rate,avg_followers_gained,median_followers_gained,avg_reach
0,False,19536,0.042232,0.0410,501.838247,501.0,6250.405764
1,True,10463,0.041875,0.0399,502.740132,501.0,6313.682883


整體比較顯示，加入 CTA 的貼文平均互動率為 4.188%，略低於未加入 CTA 的 4.223%。在新增追蹤者方面，有 CTA 的貼文平均增加 502.74 人，未加入 CTA 的貼文則為 501.84 人，兩組中位數皆為 501 人。

因此，初步結果顯示，有 CTA 的貼文互動率略低、新增追蹤者略多，但差異幅度很小，目前尚無法判定加入 CTA 是否有助於提升表現。下一步將在相同帖子形式與內容主題下進行比較，以減少內容組成差異的影響。

## 2.相同形式與主題下的 CTA 比較

In [28]:
controlled = (
    df.groupby([
        "media_type",
        "content_category",
        "has_call_to_action"
    ])
    .agg(
        posts=("post_id", "count"),
        avg_reach=("reach", "mean"),
        avg_engagement_rate=("engagement_rate", "mean"),
        avg_followers_gained=("followers_gained", "mean")
    )
    .reset_index()
)
controlled

,media_type,content_category,has_call_to_action,posts,avg_reach,avg_engagement_rate,avg_followers_gained
0,carousel,Beauty,False,710,6453.291549,0.041472,505.849296
1,carousel,Beauty,True,368,6812.839674,0.038935,518.682065
2,carousel,Comedy,False,677,6460.360414,0.042648,522.843427
3,carousel,Comedy,True,369,6361.780488,0.040898,471.436314
4,carousel,Fashion,False,664,6645.858434,0.042142,502.004518
5,carousel,Fashion,True,361,6597.240997,0.041407,488.556787
6,carousel,Fitness,False,674,6096.115727,0.041934,510.437685
7,carousel,Fitness,True,368,6208.869565,0.042960,478.383152
8,carousel,Food,False,705,6156.175887,0.041946,501.347518
9,carousel,Food,True,384,6490.622396,0.043753,486.851562


將貼文依形式與內容主題分開後，共得到 30 種內容情境。在這些情境中，有 CTA 的平均觸及較高者有 18 組、平均互動率較高者有 13 組、平均新增追蹤者較多者有 17 組。

只有 Fitness Reel 與 Travel Reel 在合併所有帳號的結果中，三項平均指標都較高。這表示 CTA 的表現並不固定，而是可能隨帖子形式與內容主題改變。不過，這個階段仍把不同帳號合併，因此下一步需要在同一帳號內比較，確認結果是否在不同帳號中一致。

## 3. 在相同帖子形式與內容主題下比較 CTA

為減少不同帳號本身差異的影響，本階段依帳號、帖子形式、內容主題及有無 CTA 進行分組，計算各組的貼文數、平均觸及、平均互動率及平均新增追蹤者。

這張摘要表將用於下一步配對同一帳號、同一形式及同一主題中的有 CTA 與沒有 CTA 貼文。

In [29]:
account_cta = (
    df.groupby(
        [
            "account_id",
            "media_type",
            "content_category",
            "has_call_to_action"
        ],
        observed=True
    )
    .agg(
        posts=("post_id", "count"),
        avg_reach=("reach", "mean"),
        avg_engagement_rate=("engagement_rate", "mean"),
        avg_followers_gained=("followers_gained", "mean")
    )
    .reset_index()
)

print(account_cta.head(10).to_string(index=False))

account_id media_type content_category  has_call_to_action  posts   avg_reach  avg_engagement_rate  avg_followers_gained
         1   carousel           Beauty               False     24 5166.083333             0.045163            378.291667
         1   carousel           Beauty                True     20 7309.550000             0.038995            621.600000
         1   carousel           Comedy               False     23 6757.652174             0.044661            603.782609
         1   carousel           Comedy                True     19 5672.052632             0.039316            493.000000
         1   carousel          Fashion               False     33 6627.393939             0.041600            515.636364
         1   carousel          Fashion                True     16 5524.562500             0.039850            460.625000
         1   carousel          Fitness               False     38 7642.578947             0.043384            505.921053
         1   carousel          F

In [30]:
keys = ["account_id", "media_type", "content_category"]

no_cta = (
    account_cta[
        account_cta["has_call_to_action"] == False
    ]
    .drop(columns="has_call_to_action")
)

yes_cta = (
    account_cta[
        account_cta["has_call_to_action"] == True
    ]
    .drop(columns="has_call_to_action")
)

paired = yes_cta.merge(
    no_cta,
    on=keys,
    suffixes=("_yes", "_no"),
    how="inner"
)

paired["reach_diff"] = (
    paired["avg_reach_yes"] - paired["avg_reach_no"]
)

paired["engagement_diff_pp"] = (
    paired["avg_engagement_rate_yes"]
    - paired["avg_engagement_rate_no"]
) * 100

paired["followers_diff"] = (
    paired["avg_followers_gained_yes"]
    - paired["avg_followers_gained_no"]
)

print("可配對的帳號與內容組合數：", len(paired))

可配對的帳號與內容組合數： 600


資料共有 20 個帳號、3 種帖子形式及 10 種內容主題，理論上共有 600 種帳號與內容情境組合。本分析成功配對全部 600 組，表示每個組合皆同時包含有 CTA 與沒有 CTA 的貼文，因此可進行帳號內比較。

第二層結論：CTA在不同內容情境中的表現

在控制帖子形式與內容主題後，共比較30種內容情境。結果顯示，有CTA的平均觸及較高者有18組、平均互動率較高者有13組、平均新增追蹤者較多者有17組。

只有Fitness Reel與Travel Reel在合併所有帳號後，三項平均指標都較高。其中，Fitness Reel有CTA時，平均觸及增加約522.33、平均互動率增加0.143個百分點、平均新增追蹤者增加約30.87人；Travel Reel則分別增加約119.54、0.029個百分點及17.70人。

結果顯示，CTA並未在所有形式與主題中呈現一致優勢，其表現可能受到內容情境影響。Fitness Reel與Travel Reel可先列為候選情境，但仍需進一步分帳號檢查，以確認結果是否在不同帳號中一致。
由於第二層仍將不同帳號的貼文合併，結果可能受到部分帳號的發文數量或原有表現影響。因此，下一階段改在同一帳號、同一形式及同一主題內比較有無CTA。

## 在同一帳號、同一形式、同一主題內比較

In [31]:
account_consistency = (
    paired.groupby(["media_type", "content_category"])
    .agg(
        paired_accounts=("account_id", "nunique"),

        median_posts_yes=("posts_yes", "median"),
        median_posts_no=("posts_no", "median"),

        reach_positive_accounts=(
            "reach_diff", lambda x: (x > 0).sum()
        ),
        engagement_positive_accounts=(
            "engagement_diff_pp", lambda x: (x > 0).sum()
        ),
        followers_positive_accounts=(
            "followers_diff", lambda x: (x > 0).sum()
        ),

        median_reach_diff=("reach_diff", "median"),
        median_engagement_diff_pp=(
            "engagement_diff_pp", "median"
        ),
        median_followers_diff=("followers_diff", "median")
    )
    .reset_index()
)

print(account_consistency.to_string(index=False))

media_type content_category  paired_accounts  median_posts_yes  median_posts_no  reach_positive_accounts  engagement_positive_accounts  followers_positive_accounts  median_reach_diff  median_engagement_diff_pp  median_followers_diff
  carousel           Beauty               20              19.0             35.5                       12                             7                           10         481.377451                  -0.189405               4.188971
  carousel           Comedy               20              18.5             32.5                        8                             6                            8        -324.529622                  -0.296885             -40.607328
  carousel          Fashion               20              17.5             33.0                        7                             8                           10        -303.090135                  -0.093239              -5.917358
  carousel          Fitness               20              19.0      

接下來我們把20個帳號分開來看，而且只比較同一帳號、同一形式、同一主題的貼文。600種組合都有成功配對，也就是每一組都有使用CTA和沒有使用CTA的貼文。結果發現，CTA並不是在每個帳號中都比較好。只有Travel主題的Carousel，以及Fitness、Food和Lifestyle主題的Reel，在三項指標上都有超過一半的帳號表現較高，像是Carousel × Travel	11/20	12/20	14/20。不過，多數情況也只有大約11到14個帳號呈現正向結果，所以一致性不算非常強。綜合前一階段的整體結果，Fitness Reel最穩定，因為不論合併所有帳號或把帳號分開，它的觸及、互動率和新增追蹤者都呈現較好的方向。因此，可以優先針對Fitness Reel進行CTA的A/B測試，但目前還不能直接說CTA一定會提升表現。

In [32]:
import pandas as pd
import numpy as np
import plotly.express as px

In [33]:
plot_df = df.copy()

plot_df["CTA"] = np.where(
    plot_df["has_call_to_action"].astype(bool),
    "有 CTA",
    "沒有 CTA"
)

overall_chart = (
    plot_df.groupby("CTA", observed=True)
    .agg(
        posts=("post_id", "count"),
        avg_reach=("reach", "mean"),
        avg_engagement_rate=("engagement_rate", "mean"),
        avg_followers_gained=("followers_gained", "mean")
    )
    .reset_index()
)

overall_chart["avg_engagement_rate_pct"] = (
    overall_chart["avg_engagement_rate"] * 100
)

display(overall_chart)

,CTA,posts,avg_reach,avg_engagement_rate,avg_followers_gained,avg_engagement_rate_pct
0,有 CTA,10463,6313.682883,0.041875,502.740132,4.187540
1,沒有 CTA,19536,6250.405764,0.042232,501.838247,4.223151


## 製作整體有無 CTA 的三張圖

In [34]:
metric_settings = [
    (
        "avg_reach",
        "有無 CTA 的平均觸及比較",
        "平均觸及"
    ),
    (
        "avg_engagement_rate_pct",
        "有無 CTA 的平均互動率比較",
        "平均互動率（%）"
    ),
    (
        "avg_followers_gained",
        "有無 CTA 的平均新增追蹤者比較",
        "平均新增追蹤者"
    )
]

overall_figs = {}

for column, title, y_title in metric_settings:

    fig = px.bar(
        overall_chart,
        x="CTA",
        y=column,
        color="CTA",
        text=column,
        title=title,
        category_orders={
            "CTA": ["沒有 CTA", "有 CTA"]
        },
        color_discrete_map={
            "沒有 CTA": "#9E9E9E",
            "有 CTA": "#4C78A8"
        }
    )

    fig.update_traces(
        texttemplate="%{text:.2f}",
        textposition="outside"
    )

    fig.update_layout(
        xaxis_title="",
        yaxis_title=y_title,
        showlegend=False
    )

    fig.show()

    overall_figs[column] = fig

## 重新建立第二層比較表
相同形式、相同主題下比較有無 CTA。

In [35]:
controlled = (
    df.groupby(
        [
            "media_type",
            "content_category",
            "has_call_to_action"
        ],
        observed=True
    )
    .agg(
        posts=("post_id", "count"),
        avg_reach=("reach", "mean"),
        avg_engagement_rate=("engagement_rate", "mean"),
        avg_followers_gained=("followers_gained", "mean")
    )
    .reset_index()
)

print("資料列數：", len(controlled))
display(controlled.head())

資料列數： 60


,media_type,content_category,has_call_to_action,posts,avg_reach,avg_engagement_rate,avg_followers_gained
0,carousel,Beauty,False,710,6453.291549,0.041472,505.849296
1,carousel,Beauty,True,368,6812.839674,0.038935,518.682065
2,carousel,Comedy,False,677,6460.360414,0.042648,522.843427
3,carousel,Comedy,True,369,6361.780488,0.040898,471.436314
4,carousel,Fashion,False,664,6645.858434,0.042142,502.004518


## 計算每種情境的 CTA 差異

In [36]:
keys = [
    "media_type",
    "content_category"
]

cta_yes = (
    controlled[
        controlled["has_call_to_action"].astype(bool)
    ]
    .drop(columns="has_call_to_action")
)

cta_no = (
    controlled[
        ~controlled["has_call_to_action"].astype(bool)
    ]
    .drop(columns="has_call_to_action")
)

content_diff = cta_yes.merge(
    cta_no,
    on=keys,
    suffixes=("_yes", "_no"),
    how="inner"
)

content_diff["reach_diff"] = (
    content_diff["avg_reach_yes"]
    - content_diff["avg_reach_no"]
)

content_diff["engagement_diff_pp"] = (
    content_diff["avg_engagement_rate_yes"]
    - content_diff["avg_engagement_rate_no"]
) * 100

content_diff["followers_diff"] = (
    content_diff["avg_followers_gained_yes"]
    - content_diff["avg_followers_gained_no"]
)

print("成功比較的內容情境數：", len(content_diff))

display(
    content_diff[
        [
            "media_type",
            "content_category",
            "reach_diff",
            "engagement_diff_pp",
            "followers_diff"
        ]
    ]
)

成功比較的內容情境數： 30


,media_type,content_category,reach_diff,engagement_diff_pp,followers_diff
0,carousel,Beauty,359.548125,-0.253760,12.832769
1,carousel,Comedy,-98.579926,-0.174961,-51.407113
2,carousel,Fashion,-48.617437,-0.073492,-13.447731
3,carousel,Fitness,112.753838,0.102620,-32.054533
4,carousel,Food,334.446509,0.180679,-14.495955
5,carousel,Lifestyle,281.075827,-0.325743,9.351714
6,carousel,Music,1.010654,-0.175896,-10.168430
7,carousel,Photography,-156.261243,0.009306,-4.802369
8,carousel,Technology,212.579101,0.253293,-31.849206
9,carousel,Travel,-44.218145,-0.021094,28.479897


正數代表有 CTA 較高，負數代表沒有 CTA 較高。

## 計算每種情境的 CTA 差異

In [37]:
def show_diff_heatmap(data, value, title, color_title, text_format):

    matrix = data.pivot(
        index="content_category",
        columns="media_type",
        values=value
    )

    fig = px.imshow(
        matrix,
        text_auto=text_format,
        aspect="auto",
        color_continuous_scale="RdYlGn",
        color_continuous_midpoint=0,
        title=title,
        labels={
            "x": "帖子形式",
            "y": "內容主題",
            "color": color_title
        }
    )

    fig.update_xaxes(side="top")
    fig.show()

    return fig

In [38]:
reach_heatmap = show_diff_heatmap(
    content_diff,
    "reach_diff",
    "不同內容情境的 CTA 觸及差異",
    "觸及差異",
    ".1f"
)

engagement_heatmap = show_diff_heatmap(
    content_diff,
    "engagement_diff_pp",
    "不同內容情境的 CTA 互動率差異",
    "百分點差異",
    ".3f"
)

followers_heatmap = show_diff_heatmap(
    content_diff,
    "followers_diff",
    "不同內容情境的 CTA 新增追蹤者差異",
    "追蹤者差異",
    ".1f"
)

熱圖中：
- 綠色：有 CTA 較高。
- 紅色：沒有 CTA 較高。
- 接近零：兩組差異很小。

## 不同帳號的一致性圖

In [39]:
consistency_long = account_consistency.melt(
    id_vars=[
        "media_type",
        "content_category",
        "paired_accounts"
    ],
    value_vars=[
        "reach_positive_accounts",
        "engagement_positive_accounts",
        "followers_positive_accounts"
    ],
    var_name="metric",
    value_name="positive_accounts"
)

consistency_long["positive_percentage"] = (
    consistency_long["positive_accounts"]
    / consistency_long["paired_accounts"]
    * 100
)

metric_names = {
    "reach_positive_accounts": "觸及",
    "engagement_positive_accounts": "互動率",
    "followers_positive_accounts": "新增追蹤者"
}

consistency_long["metric"] = (
    consistency_long["metric"].map(metric_names)
)

account_fig = px.bar(
    consistency_long,
    x="content_category",
    y="positive_percentage",
    color="metric",
    barmode="group",
    facet_col="media_type",
    title="不同帳號中有 CTA 表現較高的比例",
    labels={
        "content_category": "內容主題",
        "positive_percentage": "有 CTA 較高的帳號比例（%）",
        "metric": "指標"
    }
)

account_fig.add_hline(
    y=50,
    line_dash="dash",
    line_color="black"
)

account_fig.update_yaxes(range=[0, 100])
account_fig.update_xaxes(tickangle=45)
account_fig.show()

虛線是50%：
- 高於虛線：超過一半帳號的有 CTA 貼文較高。
- 低於虛線：多數帳號沒有呈現正向結果。

在30種「帖子形式 × 內容主題」情境中，只有以下4種情境的觸及、互動率及新增追蹤者，都在超過一半的帳號中呈現較高表現：

- Carousel Travel：觸及55%、互動率60%、新增追蹤者70%。
- Reel Fitness：觸及55%、互動率60%、新增追蹤者70%。
- Reel Food：觸及55%、互動率65%、新增追蹤者60%。
- Reel Lifestyle：觸及55%、互動率60%、新增追蹤者60%。
## CTA不是每個帳號都有效，而且可能只提升某一個指標。

## 4. 綜合結論

整體比較顯示，有 CTA 的貼文平均觸及及新增追蹤者略高，但平均互動率略低，而且差異幅度都很小。

進一步依帖子形式與內容主題比較後，30種內容情境中，有 CTA 的平均觸及較高者有18組、平均互動率較高者有13組、平均新增追蹤者較多者有17組，顯示 CTA 的表現方向會隨內容情境改變。

在同一帳號、同一形式及同一主題內比較後，只有 Carousel Travel、Reel Fitness、Reel Food及Reel Lifestyle在三項指標上皆有過半數帳號呈現較高表現，但一致程度有限。

綜合整體與帳號層級結果，Fitness Reel的表現方向最一致，因此可優先進行CTA的A/B測試。不過，本分析使用歷史資料，只能表示CTA與貼文表現之間的關聯，不能證明CTA直接造成表現提升。

In [40]:
controlled.to_csv(
    "Q2_controlled_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

content_diff.to_csv(
    "Q2_content_difference.csv",
    index=False,
    encoding="utf-8-sig"
)

account_consistency.to_csv(
    "Q2_account_consistency.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Q2分析結果已匯出")

Q2分析結果已匯出


Dashboard website

http://127.0.0.1:8050/